In [2]:
import pandas as pd
import os
import tempfile
from pathlib import Path

# 🗺️ Mapping from PROJECT SITE to REGION LWDB
region_lwdb_map = {
    "Alameda County Workforce Development Board":"Alameda County Workforce Development Board",
    "Golden Sierra Workforce Development Boards":"Golden Sierra Workforce Development Board",
    "Humboldt County Workforce Development Boards":"Humboldt County Workforce Development Board",
    "Los Angeles City Workforce Development Board":"City of Los Angeles Workforce Development Board",
    "Madera County Workforce Development Board":"Workforce Development Board of Madera County",
    "North Central Counties Consortium (NCCC)":"North Central Counties Consortium",
    "Sacramento Employment and Training Agency (SETA)": "Sacramento Employment and Training Agency",
    "San Bernardino County Workforce Development Board":"San Bernardino County Workforce Development Department",
    "San Joaquin County Workforce Development Board":"San Joaquin County Workforce Development Board",
    "Solano County Workforce Development Board":"Workforce Investment Board of Solano County",
    "South Bay Workforce Investment Board (SBWIB)":"South Bay Workforce Investment Board",
    "Southeast Los Angeles County Workforce Development Board (SELACO WDB)":"Southeast Los Angeles Workforce Development Board",
    "Ventura County Workforce Development Board":"Workforce Development Board of Ventura County",
    "Yolo County Workforce Development Board":"Yolo County Workforce Development Board",
    "Mother Lode Workforce Development Board":"Mother Lode Workforce Development Board",
    "Northern Rural Training and Employment Consortium (NoRTEC)":"Northern Rural Training and Employment Consortium",
    "Riverside County Workforce Development Board":"Riverside County Workforce Development Board",
    "San Diego Workforce Partnership, Inc.":"San Diego Workforce Partnership"

}
REGION_CODE_LOOKUP = {
    "Sacramento Employment and Training Agency":	29,
    "Workforce Investment Board of Solano County":	43,
    "Workforce Development Board of Ventura County":	48,
    "North Central Counties Consortium":	23,
    "Southeast Los Angeles Workforce Development Board":	42,
    "Workforce Development Board of Madera County":	15,
    "San Bernardino County Workforce Development Department":	32,
    "City of Los Angeles Workforce Development Board":	12,
    "Golden Sierra Workforce Development Board":	7, 
    "Yolo County Workforce Development Board":	50,
    "Humboldt County Workforce Development Board":	8,
    "South Bay Workforce Investment Board":	45,
    "San Joaquin County Workforce Development Board":	35,
    "Alameda County Workforce Development Board":	1, 
    "Mother Lode Workforce Development Board":110,
    "Northern Rural Training and Employment Consortium":120,
    "Riverside County Workforce Development Board":130,
    "San Diego Workforce Partnership":140
}
# 🔥 Definition to replace non-breaking spoaces
def normalize_spaces(series: pd.Series) -> pd.Series:
    return (
        series
        .str.replace("\u00A0", " ", regex=False)  # NBSP → normal space
        .str.replace(r"\s+", " ", regex=True)    # collapse whitespace
        .str.strip()
    )


# User home directory (dynamic)
home_dir = Path.home()

# SharePoint relative path inside OneDrive
sharepoint_relative_input_path = Path(
    "APM US",
    "Data and Insights - Documents",
    "Power BI Source Files",
    "DOR-AJCC Collab",
    "Files for Python"
)

base_dir = home_dir / sharepoint_relative_input_path

# 🔥 File for conversion
file_substring = "Phase II- Needs Assessment Responses Raw Data"

# 🧐 Get files matching substring
matching_files = [
    f for f in base_dir.glob("*.xlsx")
    if file_substring in f.name
]
# ☠️ No files matching error message
if not matching_files:
    raise FileNotFoundError("No matching Excel file found.")
    
# ☠️ Muiltiple files matching substring error message
if len(matching_files) > 1:
    raise ValueError(
        f"Multiple matching files found:\n{[f.name for f in matching_files]}"
    )

input_path = matching_files[0]

print(f"✅ Using file: {input_path}")

# ✔️ Declare path to write new file to
sharepoint_relative_output_path = Path(
    "APM US",
    "Data and Insights - Documents",
    "Power BI Source Files",
    "DOR-AJCC Collab"
)

output_base_dir = home_dir / sharepoint_relative_output_path

# ✔️ Full path with name for output file 
output_path = rf'{output_base_dir}\{file_substring}.xlsx'
# output_path = rf'C:\Users\StevenFoster\Downloads\{file_name}.xlsx'

# 🛻 Load the 1st worksheet with NO headers
df_raw = pd.read_excel(input_path, sheet_name=0, header=None)

# 🛻 Find the row index where column 0 == "Project Site"
header_row_idx = df_raw.index[df_raw.iloc[:, 0] == "Project Site"]

# ☠️ Error message when column header not located
if header_row_idx.empty:
    raise ValueError("Header row containing 'Project Site' not found.")

header_row_idx = header_row_idx[0]

# ➖ Remove rows above the header
df_clean = df_raw.iloc[header_row_idx:].reset_index(drop=True)

# 🛻 Promote that row to headers
df_clean.columns = df_clean.iloc[0]
df_clean = df_clean.iloc[1:].reset_index(drop=True)

# ✔️ Clean column names
df_clean.columns = (
    df_clean.columns
        .astype(str)
        .str.replace('\n', ' ', regex=False)  # replace newline with space
        .str.replace(r'\s+', ' ', regex=True) # collapse multiple spaces
        .str.strip()                           # trim leading/trailing spaces
)

# ➖ Remove repeated header rows inside the data
df_clean = df_clean[df_clean["Project Site"] != "Project Site"]

# ➖ Remove total row
df_clean = df_clean[~df_clean["Project Site"].astype(str).str.contains(
    "Total", na=False
)]


# 🤖 Capitlize headers
df_clean.columns = df_clean.columns.str.upper()

# 4️⃣ Validate columns are present 
expected_string_cols = [
    "PROJECT SITE",
    "SYSTEMS USED",
    "ADDITIONAL FEEDBACK",
    "RISKS",
    "PROCESS GAPS",
    "CO-ENROLLED PROGRAMS",
    "OTHER CO-ENROLLED PROGRAMS",
    "TRACKING METHOD",
    "COORDINATION METHODS",
    "BRAIDED FUNDING", 
    "OTHER BRAIDED FUNDING",
    "COMPLIANCE COMPONENTS",
    "ENROLLMENT BARRIERS",
    "OTHER ENROLLMENT BARRIERS",
    "FIRST PROGRAM ENROLLED",
    "OTHER FIRST PROGRAM ENROLLED",
    "TRACKING METHOD",
    "AJCC–DOR ALIGNMENT",
    "BOTTLENECK IDENTIFICATION",
    "CASE NOTES USAGE",
    "CO-CASE MANAGEMENT",
    "CONFIDENCE",
    "COORDINATION FREQUENCY",
    "DOCUMENTED PROCESS",
    "GOAL FEASIBILITY",
    "PROGRESS STATUS",
    "READINESS TIMELINE",
    "SCOPE CHANGES",
    "SCREENING CRITERIA",
    "OTHER SCREENING CRITERIA",
    "STAFF AWARENESS",
    "TIMING OF ENROLLMENT",
    "TRAINING",
    "WORKFLOW EFFECTIVENESS",
    "WORKFLOW ESTABLISHED",
    "WORKFLOW VISIBILITY", 
    "TA NEEDS"
]

for col in expected_string_cols:
    if col not in df_clean.columns:
        raise ValueError(f"❌ Missing expected string column: '{col}'")

    # Convert to string
    df_clean[col] = df_clean[col].astype("string")

    # Verify all values are strings (or NA)
    bad_mask = df_clean[col].apply(lambda v: not (pd.isna(v) or isinstance(v, str)))
    if bad_mask.any():
        bad_values = df_clean.loc[bad_mask, col].head(10)
        raise TypeError(
            f"❌ Column '{col}' contains non-string data.\n"
            f"Example invalid values:\n{bad_values}"
        )

print("✅ All column types validated successfully. Safe to continue.")


# ➕ Define what columns should be kept the 1st worksheet with NO headers
cols_to_keep = [
    "PROJECT SITE",
    "SYSTEMS USED",
    "ADDITIONAL FEEDBACK",
    "RISKS",
    "PROCESS GAPS",
    "CO-ENROLLED PROGRAMS",
    "OTHER CO-ENROLLED PROGRAMS",
    "TRACKING METHOD",
    "COORDINATION METHODS",
    "BRAIDED FUNDING", 
    "OTHER BRAIDED FUNDING",
    "COMPLIANCE COMPONENTS",
    "ENROLLMENT BARRIERS",
    "OTHER ENROLLMENT BARRIERS",
    "FIRST PROGRAM ENROLLED",
    "OTHER FIRST PROGRAM ENROLLED",
    "AJCC–DOR ALIGNMENT",
    "BOTTLENECK IDENTIFICATION",
    "CASE NOTES USAGE",
    "CO-CASE MANAGEMENT",
    "CONFIDENCE",
    "COORDINATION FREQUENCY",
    "DOCUMENTED PROCESS",
    "GOAL FEASIBILITY",
    "PROGRESS STATUS",
    "READINESS TIMELINE",
    "SCOPE CHANGES",
    "SCREENING CRITERIA",
    "OTHER SCREENING CRITERIA",
    "STAFF AWARENESS",
    "TIMING OF ENROLLMENT",
    "TRAINING",
    "WORKFLOW EFFECTIVENESS",
    "WORKFLOW ESTABLISHED",
    "WORKFLOW VISIBILITY", 
    "TA NEEDS"
]

df_stripped = df_clean[cols_to_keep].copy()

# columns that should be split on newline
expand_cols = [
    "CO-ENROLLED PROGRAMS",
    "OTHER CO-ENROLLED PROGRAMS",
    "TRACKING METHOD",
    "COORDINATION METHODS",
    "BRAIDED FUNDING", 
    "OTHER BRAIDED FUNDING",
    "COMPLIANCE COMPONENTS",
    "ENROLLMENT BARRIERS",
    "OTHER ENROLLMENT BARRIERS",
    "FIRST PROGRAM ENROLLED",
    "OTHER FIRST PROGRAM ENROLLED",
    "AJCC–DOR ALIGNMENT",
    "BOTTLENECK IDENTIFICATION",
    "CASE NOTES USAGE",
    "CO-CASE MANAGEMENT",
    "CONFIDENCE",
    "COORDINATION FREQUENCY",
    "DOCUMENTED PROCESS",
    "GOAL FEASIBILITY",
    "PROGRESS STATUS",
    "READINESS TIMELINE",
    "SCOPE CHANGES",
    "SCREENING CRITERIA",
    "OTHER SCREENING CRITERIA",
    "STAFF AWARENESS",
    "TIMING OF ENROLLMENT",
    "TRAINING",
    "WORKFLOW EFFECTIVENESS",
    "WORKFLOW ESTABLISHED",
    "WORKFLOW VISIBILITY", 
    "TA NEEDS"
]

# identifier columns that should NOT be repeated unnecessarily
id_cols = ["PROJECT SITE", 
           "SYSTEMS USED", 
           "ADDITIONAL FEEDBACK", 
           "RISKS",
           "PROCESS GAPS"       
          ]

long_df = df_stripped.melt(
    id_vars=id_cols,
    var_name="QUESTION",
    value_name="RESPONSE"
)

# normalize line breaks
long_df["RESPONSE"] = (
    long_df["RESPONSE"]
        .astype("string")
        .str.replace("\r\n", "\n")
        .str.replace("\r", "\n")
)

# flag rows that should be split
split_mask = long_df["QUESTION"].isin(expand_cols)

# split and explode only those rows
expanded = (
    long_df[split_mask]
        .assign(RESPONSE=lambda x: x["RESPONSE"].str.split("\n"))
        .explode("RESPONSE")
)
tracking_values = [
    "Documented primarily through case notes",
    "Entered and tracked directly in CalJOBS",
    "Tracked through shared spreadsheets or internal trackers",
    "Managed across multiple disconnected methods",
]

# collapse multiple rows into one per site
tracking_collapsed = (
    long_df[long_df["QUESTION"] == "TRACKING METHOD"]
    .groupby(id_cols + ["QUESTION"])["RESPONSE"]
    .apply(lambda x: "\n".join(x.dropna().unique()))
    .reset_index()
)

# keep non-expanded rows as-is
non_expanded = long_df[~split_mask]

# recombine
final_df = pd.concat([expanded, non_expanded], ignore_index=True)

# 🤖 Drop blank rows, fail back for "used range" in excel file
final_df = final_df.dropna(how="all")

# ➕ Add REGION LWDB column with fallback to PROJECT SITE
final_df["REGION LWDB"] = final_df["PROJECT SITE"].map(region_lwdb_map)

# ➕ Map REGION CODE
final_df["REGION CODE"] = final_df["REGION LWDB"].map(REGION_CODE_LOOKUP)

# 🤖 Trim all string columns 
for col in final_df.select_dtypes(include="string").columns:
    final_df[col] = normalize_spaces(final_df[col])


# 🤖 Convert output path to pathlib path 
output_path = Path(output_path)

# 🤖 Check if file exists and remove 
if output_path.exists():
    print(f"⚠️ Overwriting existing file: {output_path}")
    output_path.unlink()  # deletes the file
else: 
    print(f"🆕 File does not exist. Creating new file at: {output_path}")
    
# ✍️ Export to Excel
final_df.to_excel(output_path, index=False)

✅ Using file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\DOR-AJCC Collab\Files for Python\4.23.26 Phase II- Needs Assessment Responses Raw Data.xlsx
✅ All column types validated successfully. Safe to continue.
⚠️ Overwriting existing file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\DOR-AJCC Collab\Phase II- Needs Assessment Responses Raw Data.xlsx
